# Prepare Excel file for sequence submission 
Krista Longnecker, 3 September 2026\
Updated 11 September 2026 to use multiple plates
Updated 18 September 2026 to use cruise/cast/niskin from BATS cast sheets

Have an Excel file that will go to Dalhousie, need to make a sheet for BIOS-SCOPE-specific details. The Excel file template is from Dalhousie's IMR group

In [62]:
%reset -f

In [63]:
import os
import re
import pandas as pd
import xlwings as xw #need to use this to make the new sheet to keep the Dalhousie validations
import pdb #use with set_trace()

In [64]:
data_dir = 'seq_data/'
os.listdir(data_dir)

['10412_biosscope.xlsx',
 '10414_biosscope.xlsx',
 '10415_biosscope.xlsx',
 '10416_biosscope.xlsx',
 '10417_biosscope.xlsx',
 '10418_biosscope.xlsx',
 '10419_biosscope.xlsx',
 '10420_biosscope.xlsx',
 '10421_biosscope.xlsx',
 '10422_biosscope.xlsx',
 '10423_biosscope.xlsx',
 '10424_biosscope.xlsx',
 '10425_biosscope.xlsx',
 '10426_biosscope.xlsx',
 '10427_biosscope.xlsx',
 '10428_biosscope.xlsx',
 '10429_biosscope.xlsx',
 '10430_biosscope.xlsx',
 '20413_biosscope.xlsx',
 '20414_biosscope.xlsx',
 '20415_biosscope.xlsx',
 '20423_biosscope.xlsx',
 'biosscope_combinedCCN.xlsx',
 'Dalhousie BIOS-SCOPE DNA Submission Test2.2026.09.19.xlsx',
 'dups.xlsx',
 'IMR-SubmissionTemplate_BIOSSCOPE_13092026.xlsx',
 'IMR-SubmissionTemplate_BIOSSCOPE_2026.09.19.xlsx',
 'IMR-SubmissionTemplate_BIOSSCOPE_2026.09.23.xlsx',
 'oldFiles',
 'Sample_sheets_20260915',
 'templx.xlsx']

In [65]:
# keep these here - useful to track down path and filename issues
# os.getcwd()
# os.listdir('seq_data')

In [66]:
fName = 'IMR-SubmissionTemplate_BIOSSCOPE_2026.09.23.xlsx'
sheet = 'Amplicon - MiSeq'

# fName = 'Dalhousie BIOS-SCOPE DNA Submission Test2.xlsx'
# sheet = 'Dalhousie Submission'

Can I use the BATS cast sheets to make a NewID (need cruise / cast / Niskin)?

In [67]:
#set this here as I will know at this point if I have what I need from BATS
possible = 1 #have what we need from BATS
# possible = 0 #do NOT have what we need from BATS

Probably (hopefully) do not need to make any changes below here.

In [68]:
# make my NewID (or a placeholder if I don't have cast and Niskin information)
def makeID(possible,row):
    #pdb.set_trace()
    #do we have the data we need from BATS to determine cast and Niskin information?
    if possible == 1:
        one = row['cruise5'] + f"{row['cast']:02d}" + f"{row['niskin']:02d}"
    else:
        one = row['cruise5'] + 'ccc' + 'nn'
    
    return one

In [69]:
#parsing out the information in the filenames, use a function
def parseSample(sample):
    #parsed = {} #use the next row and setup the dictionary first
    #even empty it is useful as I don't have to repeat all the NA options
    parsed = dict({'cruise5':'',
                   'cast':'',
                   'niskin':'',
                   'cruise':'',
                   'otherInfo':'',
                   'depth':'',
                   'sampleNumber':'',
                   'New_Bottle_ID':'',
                   'seqType':''})
    #start with 10 digits - cruise - cast - niskin and then underscores to note 18S_V4
    ru = [match.start() for match in re.finditer("_",sample)]
    #print(sample)
    if len(ru) ==2 & ru[0]==10:
        #pdb.set_trace()
        parsed['cruise5'] = sample[:5]
        parsed['cast'] = sample[5:8]
        parsed['niskin']= sample[8:10]        
        parsed['New_Bottle_ID'] = sample[:10]
        parsed['otherInfo'] = sample[ru[0]+1:]
        if '18S' in parsed['otherInfo']:
            parsed['seqType'] = 'V4_18s'
        elif '16S_V1V2' in parsed['otherInfo']:
            parsed['seqType'] = 'V1V2'
    elif (len(ru) ==2) & (ru[0] ==5):
        parsed['cruise5'] = sample[:5]
        parsed['sampleNumber'] = sample[ru[0]+1 : ru[1]]
        parsed['depth'] = sample[ru[1]+1 : -1]
    elif (len(ru) ==1):
        parsed['cruise5'] = sample[:5]
        parsed['depth'] = sample[ru[0]+1 : -1]
        
            
    parsed = pd.DataFrame([parsed]) 
    return parsed

In [70]:
# 1st box with names is B31 to M38 
chunks = 'B:M,P:AA,AD:AO' #add other plates as needed here
df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName),sheet_name = sheet,usecols = chunks,skiprows = 30, nrows = 8,header = None,dtype=str))

#the older data files used hyphens and not underscore...
df = df.replace('-','_',regex = True)
df

,1,2,3,4,5,6,7,8,9,10,...,31,32,33,34,35,36,37,38,39,40
0,20413_1_0m,10414_1_0m,10414_9_500m,10415_5_160m,20415_1_0m,10416_1_0m,10416_9_500m,10417_5_160m,10418_1_0m,10418_9_500m,...,50064_019_500m,50064_015_165m,50064_013_300m,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20413_2_40m,10414_2_40m,10414_10_600m,10415_6_200m,20415_2_40m,10416_2_40m,10416_10_600m,10417_6_200m,10418_2_40m,10418_10_600m,...,50064_019_800m,50064_015_200m,50064_013_400m,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20413_3_80m,10414_3_80m,10414_11_800m,10415_7_250m,20415_3_80m,10416_3_80m,10416_11_800m,10417_7_250m,10418_3_80m,10418_11_800m,...,50064_017_165m,50064_015_300m,50064_013_500m,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20413_4_120m,10414_4_120m,10414_12_1000m,10415_8_300m,20415_4_120m,10416_4_120m,10416_12_1000m,10417_8_300m,10418_4_120m,10418_12_1000m,...,50064_017_200m,50064_015_400m,50064_013_820m,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20413_5_160m,10414_5_160m,10415_1_0m,10415_9_500m,20415_5_160m,10416_5_160m,10417_1_0m,10417_9_500m,10418_5_160m,10419_1_0m,...,50064_017_300m,50064_015_500m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,20413_6_200m,10414_6_200m,10415_2_40m,10415_10_600m,20415_6_200m,10416_6_200m,10417_2_40m,10417_10_600m,10418_6_200m,10419_2_40m,...,50064_017_400m,50064_015_850m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,20413_7_250m,10414_7_250m,10415_3_80m,10415_11_800m,20415_7_250m,10416_7_250m,10417_3_80m,10417_11_800m,10418_7_250m,10419_3_80m,...,50064_017_500m,50064_013_165m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,20413_8_300m,10414_8_300m,10415_4_120m,10415_12_1000m,NaN,10416_8_300m,10417_4_120m,10417_12_1000m,10418_8_300m,10419_4_120m,...,50064_017_820m,50064_013_200m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [71]:
#make one column and put sampleName for the first column's header
df = df.stack().reset_index(drop = True).to_frame()
df = df.rename(columns= {0: 'sampleName'})
#drop the NaNs
df = df.dropna(subset=['sampleName']).reset_index(drop=True)

In [72]:
#for now, trim the samples that begin with DH (Devil's Hole) and 5 (BVAL, though I am not sure if they will come back into play later)
df = df[~df['sampleName'].str.startswith(('DH','50'))]

In [73]:
df.head()

,sampleName
0,20413_1_0m
1,10414_1_0m
2,10414_9_500m
3,10415_5_160m
4,20415_1_0m


In [74]:
for idx,row in df.iterrows():
    sample = row['sampleName']
    parsed = parseSample(sample)
    df.at[idx,'cruise5'] = parsed.loc[0,'cruise5']
    df.at[idx,'sampleNumber'] = parsed.loc[0,'sampleNumber']        
    df.at[idx,'depth'] = int(parsed.loc[0,'depth'])
        
#make depth and sampleNumber both a number
df['depth'] = df['depth'].astype('Int64')

#have to add pd.NA first bc this breaks when that column is empty
df["sampleNumber"] = df['sampleNumber'].replace(r"^\s*$", pd.NA, regex=True)
df['sampleNumber'] = df['sampleNumber'].astype('Int64')
df = df.sort_values(by = ['cruise5','sampleNumber']).reset_index(drop = True)

df['NewID'] = None

In [75]:
df

,sampleName,cruise5,sampleNumber,depth,NewID
0,10414_1_0m,10414,1,0,None
1,10414_2_40m,10414,2,40,None
2,10414_3_80m,10414,3,80,None
3,10414_4_120m,10414,4,120,None
4,10414_5_160m,10414,5,160,None
...,...,...,...,...,...
105,20415_3_80m,20415,3,80,None
106,20415_4_120m,20415,4,120,None
107,20415_5_160m,20415,5,160,None
108,20415_6_200m,20415,6,200,None


In [76]:
#here - need to read in the Excel file with c/c/n (and now I realize that it is inconvenient to have one file per cast)
dfCCN = pd.read_excel(os.path.join(data_dir,'biosscope_combinedCCN.xlsx'))

In [77]:
dfCCN.head()

,NewID,BS_eDNA,Depth,Flag,cruise
0,1041201101,1,1,1,10412
1,1041201103,2,40,1,10412
2,1041201105,3,80,1,10412
3,1041201108,4,120,1,10412
4,1041201110,5,160,1,10412


In [78]:
for idx,row in df.iterrows():
    #set possible above so I don't forget
    if possible == 1:
        #find the row that matches cruise and sample number...use that to assign NewID
        i = dfCCN[(dfCCN['cruise'] ==int(row['cruise5'])) & (dfCCN['BS_eDNA'] ==row['sampleNumber'])].index[0]
        df.at[idx,'NewID'] = dfCCN.at[i,'NewID'] 
        del(i)
    elif possible == 0:
        df.at[idx,'NewID'] = makeID(possible,row) #makeID is the function defined above   

In [79]:
df.head()

,sampleName,cruise5,sampleNumber,depth,NewID
0,10414_1_0m,10414,1,0,1041401201
1,10414_2_40m,10414,2,40,1041401203
2,10414_3_80m,10414,3,80,1041401205
3,10414_4_120m,10414,4,120,1041401208
4,10414_5_160m,10414,5,160,1041401210


In [80]:
#add the columns to be filled in later:
toFill = 'fill in when sequencing is complete'
df['gzipFilename_1'] = None
df['gzipFilename_2'] = None
df['gzipFilename_3'] = None
df['seqStorageLocation'] = None

df.loc[0,['gzipFilename_1','gzipFilename_2','gzipFilename_3','seqStorageLocation']] = toFill

In [81]:
# Use xlwings to put the new sheet in the Excel file
with xw.App(visible=False) as app:
    wb = xw.Book(os.path.join(data_dir,fName))
    
    #add the BIOS-SCOPE details to the worksheet
    new_ws = wb.sheets.add(name="projectDetails2", after=wb.sheets[-1])
    new_ws.range("A1").options(index=False).value = df
    
    #add a blank sheet so I can move the snap over to there
    new_ws = wb.sheets.add(name="miscSubmissionNotes", after=wb.sheets[-1])
    
    
    wb.save()
    wb.close()

Below here will not run and is a holding zone for misc. code bits

In [82]:
#Stick some code below this spot as a holding zone
raise SystemExit("Stop execution here")

SystemExit: Stop execution here

C:\Users\LaneyLab\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3465: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
dfm = df.merge(bn,on = 'sampleNumber',how = 'left')
dfm = dfm.sort_values(by = ['cruise5','depth_x']).reset_index(drop= True) #for now keep both depth columns
dfm

In [ ]:
#do NOT do this...it does vary by cruise so I don't want to guess




## now, how to match to sample numbers...these are fixed for BIOS-SCOPE casts (still do not have access to the BATS cast sheets)
#make a lookup table:
bn = pd.DataFrame({
    'sampleNumber':range(1,13,1),
    'niskin': [1,3,5,8,10,11,13,15,17,19,21,23],
    'depth':[1,40,80,120,160,200,250,300,500,600,800,1000]})

bn['sampleNumber'] = bn['sampleNumber'].astype('Int64')
df['sampleNumber'] = df['sampleNumber'].astype('Int64')
 

In [ ]:
import openpyxl

# 1. Load the workbook and keep VBA macros
file_path = os.path.join(data_dir,'Submissions/testing.xlsx')
wb = openpyxl.load_workbook(file_path, keep_vba=True)

# 2. Add a new worksheet (specify name and optional position index)
# index=0 forces it to be the first sheet; omit index to append to the end.
new_ws = wb.create_sheet(title="NewDataSheet", index=0)

# 3. Add data to your new sheet
new_ws = df

# 4. Save the file with the macro-enabled extension
wb.save(file_path)

In [ ]:
# #put the sample information into the existing Excel file being setup for Dalhousie in append mode ('a') 
# with pd.ExcelWriter(os.path.join(data_dir,fName), mode='a', engine='openpyxl') as writer:
#     # put it in as a new sheet
#     df.to_excel(writer, sheet_name='projectDetails', index=False)